# Бие даалт 2 — Deep Knowledge Tracing-ийг хэрэгжүүлэх

**F.CSM308 Машин сургалт** · B231910012, Б. Өсөхбаяр

Үндэслэсэн өгүүлэл: C. Piech нар, *Deep Knowledge Tracing*, NeurIPS 2015 ([arXiv:1506.05908](https://arxiv.org/abs/1506.05908)).
Онолын тайлбар нь Бие даалт 1-ийн тайланд бий.

**Энэ notebook юу хийх вэ**

| # | Хэсэг | Асуулт |
|---|---|---|
| 1 | Өгөгдөл | Зохиогчдын ASSISTments 2009 файл яг юу агуулдаг вэ? Өгүүллийн Хүснэгт 1-тэй таарах уу? |
| 2 | Суурь загварууд | Marginal, BKT, PFA (логистик регресс) ямар AUC өгөх вэ? |
| 3 | DKT | LSTM ба энгийн RNN-ийг PyTorch-оор өөрөө бичиж сургах |
| 4 | Үр дүн | Өгүүллийн 0.86 AUC-г давтаж чадах уу? |
| 5 | Нэмэлт судалгаа | (a) Өгөгдлийн хэмжээ: DKT хэдэн хариултаас суурь загваруудыг давдаг вэ? (b) Бодлого хоорондын нөлөө $J_{ij}$ (c) Нэг сурагчийн мэдлэгийн явц |

**Runtime:** Colab → `Runtime` → `Change runtime type` → **T4 GPU**. CPU дээр ч ажиллана, удаан.

In [ ]:
import os, time, math, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# QUICK=1 үед бүх туршилтыг товчилж ажиллуулна (кодоо шалгахад)
QUICK = os.environ.get("DKT_QUICK") == "1"
print("device:", DEVICE, "| torch", torch.__version__, "| QUICK =", QUICK)

## 1. Өгөгдөл

Зохиогчдын [GitHub репо](https://github.com/chrispiech/DeepKnowledgeTracing)-д нийтэлсэн ASSISTments 2009–2010 *skill builder* өгөгдлийг шууд ашиглана.
Файлын бүтэц: сурагч бүрт 3 мөр.

```
7                     ← хариултын тоо
82,82,82,82,82,82,82  ← чадварын (skill) таг q_t
0,0,0,0,0,0,0         ← зөв (1) / буруу (0) a_t
```

In [ ]:
DATA_DIR = "dkt_data"
BASE = "https://raw.githubusercontent.com/chrispiech/DeepKnowledgeTracing/master/data/assistments/"
os.makedirs(DATA_DIR, exist_ok=True)
for name in ["builder_train.csv", "builder_test.csv"]:
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE + name, path)

def load_dkt_csv(path):
    # Сурагч бүрийг (q, a) хос numpy массив болгоно
    with open(path, encoding="utf-8") as f:
        lines = [l.strip().rstrip(",") for l in f]
    seqs = []
    for i in range(0, len(lines) - 2, 3):
        n = int(lines[i])
        q = np.array([int(x) for x in lines[i + 1].split(",") if x], dtype=np.int64)
        a = np.array([int(x) for x in lines[i + 2].split(",") if x], dtype=np.int64)
        assert len(q) == len(a) == n
        seqs.append((q, a))
    return seqs

train_all = load_dkt_csv(os.path.join(DATA_DIR, "builder_train.csv"))
test_all  = load_dkt_csv(os.path.join(DATA_DIR, "builder_test.csv"))
M = 1 + max(int(q.max()) for q, _ in train_all + test_all)   # чадварын тоо

def describe(seqs):
    lens = np.array([len(q) for q, _ in seqs])
    a = np.concatenate([a for _, a in seqs])
    return dict(сурагч=len(seqs), хариулт=int(lens.sum()), зөв_хувь=round(a.mean(), 3),
                урт_медиан=int(np.median(lens)), урт_max=int(lens.max()), урт_1=int((lens < 2).sum()))

stats = pd.DataFrame({"train": describe(train_all), "test": describe(test_all),
                      "нийт": describe(train_all + test_all)}).T
print("Чадварын тоо M =", M)
stats

**Өгүүлэлтэй харьцуулбал.** Өгүүллийн Хүснэгт 1-д ASSISTments-д *15,931 сурагч, 124 таг, 526K хариулт* гэж бичсэн.
Зохиогчдын өөрсдийн нийтэлсэн файлд хариултын тоо (≈525.5K) ба таг (124) таарч байгаа ч сурагчийн тоо ~4.2K байна.
Хэрэв нэг сурагчийн дарааллыг чадвар бүрээр тасалж тоолсон бол 15.9K гарч магадгүй, гэхдээ өгүүлэлд энэ талаар тайлбар байхгүй.
Иймээс өгүүллийн тоог шууд давтаж шалгах боломжгүй гэдгийг тэмдэглэж, цааш нь **файлын бодит бүтцээр** ажиллана.

In [ ]:
lens = np.array([len(q) for q, _ in train_all + test_all])
skill_cnt = np.bincount(np.concatenate([q for q, _ in train_all + test_all]), minlength=M)
skill_acc = (np.bincount(np.concatenate([q for q, _ in train_all + test_all]),
                         weights=np.concatenate([a for _, a in train_all + test_all]), minlength=M)
             / np.maximum(skill_cnt, 1))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(lens, bins=np.logspace(0, np.log10(lens.max()), 40)); ax[0].set_xscale("log")
ax[0].set_title("Сурагчийн дарааллын урт"); ax[0].set_xlabel("хариултын тоо (log)")
order = np.argsort(-skill_cnt)
ax[1].bar(range(M), skill_cnt[order]); ax[1].set_yscale("log")
ax[1].set_title("Чадвар тус бүрийн хариултын тоо"); ax[1].set_xlabel("чадвар (их → бага)")
ax[2].hist(skill_acc[skill_cnt > 0], bins=25)
ax[2].set_title("Чадвар тус бүрийн зөв хувь"); ax[2].set_xlabel("зөв хувь")
plt.tight_layout(); plt.show()

In [ ]:
# Дараалсан алхмууд ижил чадвартай байх хувь: skill builder-т сурагч нэг чадварыг
# дараалан (blocking) бодож, тодорхой тоогоор зөв бодсон үед дараагийнх руу шилждэг.
qq = [q for q, _ in train_all + test_all if len(q) >= 2]
aa = [a for q, a in train_all + test_all if len(q) >= 2]
same_q = np.mean(np.concatenate([q[1:] == q[:-1] for q in qq]))
same_qa = np.mean(np.concatenate([(q[1:] == q[:-1]) & (a[1:] == a[:-1]) for q, a in zip(qq, aa)]))
print(f"Өмнөх алхамтай ижил чадвар: {same_q:.1%} · ижил (чадвар, хариулт): {same_qa:.1%}")

Алхмуудын ихэнх нь өмнөх алхмын чадварыг давтдаг. Иймээс энэ өгөгдөл дээр **«сүүлийн хариултаа давтана»** гэх мэт энгийн дүрэм ч өндөр AUC өгөх төлөвтэй. Энэ нь доорх BKT-ийн үр дүнг тайлбарлахад чухал.

**Train / validation / test.** Зохиогчдын train/test хуваалтыг хадгална. Train-ээс 10% сурагчийг validation (early stopping) болгон салгана.
Ганц хариулттай сурагчид таамаглах алхам байхгүй тул хасна.

In [ ]:
rng = np.random.default_rng(SEED)
train_all = [s for s in train_all if len(s[0]) >= 2]
test_seqs = [s for s in test_all if len(s[0]) >= 2]
perm = rng.permutation(len(train_all))
n_val = len(train_all) // 10
val_seqs   = [train_all[i] for i in perm[:n_val]]
train_seqs = [train_all[i] for i in perm[n_val:]]
n_ans = lambda S: sum(len(q) for q, _ in S)
print(f"train {len(train_seqs)} сурагч / {n_ans(train_seqs)} хариулт · "
      f"val {len(val_seqs)} / {n_ans(val_seqs)} · test {len(test_seqs)} / {n_ans(test_seqs)}")

## 2. Суурь загварууд

Бүх загварыг **ижил зорилгоор** үнэлнэ: сурагчийн эхний хариултаас бусад бүх хариулт $a_{t+1}$-ийг өмнөх түүхээр таамаглаж, test дээр AUC тооцно.

* **Marginal** — чадвар бүрийн train дээрх дундаж зөв хувь (өгүүллийн суурь загвар).
* **BKT** — чадвар бүрт 4 параметр $(P(L_0), T, G, S)$. Параметрийг train дээр log-likelihood-ийг хамгийн их болгох **grid search**-ээр олно (өгүүллийн BKT нь EM-ээр сургагдсан байж болно; энэ нь миний хялбаршуулалт).
* **PFA** (Performance Factors Analysis, Pavlik нар 2009) — логистик регресс:
  $\;\operatorname{logit} P(a=1) = \beta_q + \gamma_q \log(1+s_q) + \rho_q \log(1+f_q)$, энд $s_q, f_q$ нь тухайн чадварт өмнө хийсэн зөв ба буруу хариултын тоо.

In [ ]:
def targets(seqs):
    # Таамаглах ёстой (q_{t+1}, a_{t+1}) хосууд: эхний хариултыг алгасна
    q = np.concatenate([q[1:] for q, _ in seqs]); a = np.concatenate([a[1:] for _, a in seqs])
    return q, a

# ---------- Marginal ----------
def marginal_predict(train, test):
    q_tr = np.concatenate([q for q, _ in train]); a_tr = np.concatenate([a for _, a in train])
    cnt = np.bincount(q_tr, minlength=M); cor = np.bincount(q_tr, weights=a_tr, minlength=M)
    p_skill = (cor + 1) / (cnt + 2)                      # Laplace жигдрүүлэлт
    q_te, a_te = targets(test)
    return a_te, p_skill[q_te]

# ---------- PFA ----------
def pfa_features(seqs, skip_first=True):
    rows, cols, vals, y = [], [], [], []
    r = 0
    for q, a in seqs:
        s = np.zeros(M); f = np.zeros(M)
        for t in range(len(q)):
            if t > 0 or not skip_first:
                k = q[t]
                rows += [r, r, r]; cols += [k, M + k, 2 * M + k]
                vals += [1.0, np.log1p(s[k]), np.log1p(f[k])]
                y.append(a[t]); r += 1
            if a[t]: s[q[t]] += 1
            else:    f[q[t]] += 1
    X = sparse.csr_matrix((vals, (rows, cols)), shape=(r, 3 * M))
    return X, np.array(y)

def pfa_predict(train, test):
    X_tr, y_tr = pfa_features(train, skip_first=False)
    X_te, y_te = pfa_features(test)
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(X_tr, y_tr)
    return y_te, clf.predict_proba(X_te)[:, 1]

In [ ]:
# ---------- BKT ----------
import itertools
GRID = np.array([p for p in itertools.product(
    [0.1, 0.3, 0.5, 0.7, 0.9],      # P(L0)
    [0.02, 0.05, 0.1, 0.2, 0.35],   # T  (сурах)
    [0.1, 0.2, 0.3],                # G  (таамаглах)
    [0.05, 0.1, 0.2, 0.3])])        # S  (алдах)

EPS = 1e-6
def bkt_update(L, o, G, S):
    # p = P(зөв); 1-p-г (1-L)(1-G) + L·S хэлбэрээр шууд тооцно.
    # 1 - p гэж хасвал L≈1 үед бөөрөнхийлөлтийн алдаа буруу хариулт бүрт ~(1-S-G)/S дахин өсөж,
    # P(L) > 1, p < 0 болдог (туршихад илэрсэн алдаа).
    p = L * (1 - S) + (1 - L) * G
    q = (1 - L) * (1 - G) + L * S
    post = L * (1 - S) / p if o else L * S / q
    return (p if o else 1 - q), post

def per_skill_runs(seqs):
    # Сурагч бүрийн дарааллыг чадвар тус бүрийн дэд дараалал болгоно (BKT чадваруудыг тусад нь үздэг)
    runs = {k: [] for k in range(M)}
    for q, a in seqs:
        for k in np.unique(q):
            runs[k].append(a[q == k])
    return runs

def bkt_fit(train):
    params = np.tile([0.5, 0.1, 0.2, 0.1], (M, 1))
    for k, runs in per_skill_runs(train).items():
        if not runs:
            continue
        L0, T, G, S = GRID.T
        ll = np.zeros(len(GRID))
        for obs in runs:                         # grid-ийн бүх утгыг нэг дор векторжуулж тооцно
            L = L0.copy()
            for o in obs:
                p, post = bkt_update(L, o, G, S)
                ll += np.log(p if o else 1 - p)
                L = np.clip(post + (1 - post) * T, EPS, 1 - EPS)
        params[k] = GRID[np.argmax(ll)]
    return params

def bkt_predict(train, test):
    params = bkt_fit(train)
    y, p_out = [], []
    for q, a in test:
        Lk = params[:, 0].copy()                       # чадвар бүрийн одоогийн P(L)
        for t in range(len(q)):
            k = q[t]; _, T, G, S = params[k]
            p = Lk[k] * (1 - S) + (1 - Lk[k]) * G        # хариулахаас өмнөх таамаглал
            if t > 0:
                y.append(a[t]); p_out.append(p)
            _, post = bkt_update(Lk[k], a[t], G, S)
            Lk[k] = min(max(post + (1 - post) * T, EPS), 1 - EPS)
    return np.array(y), np.array(p_out)

In [ ]:
results = {}
for name, fn in [("Marginal", marginal_predict), ("PFA", pfa_predict), ("BKT", bkt_predict)]:
    t0 = time.time()
    y, p = fn(train_seqs + val_seqs, test_seqs)
    results[name] = roc_auc_score(y, p)
    print(f"{name:9s} test AUC = {results[name]:.4f}   ({time.time() - t0:.0f} c)")

## 3. Deep Knowledge Tracing

### 3.1 Оролтын бэлтгэл

* $(q_t, a_t)$ хосыг нэг индекс болгоно: $x_t = q_t + M \cdot a_t \in \{0, \dots, 2M-1\}$. Өгүүллийн шиг үүнийг `F.one_hot`-оор $2M$ урттай вектор болгоно.
* Загварт $x_1..x_{T-1}$-г өгч, алхам бүрт **дараагийн** бодлого $q_{t+1}$ ба хариулт $a_{t+1}$-ийг зорилт болгоно.
* **Сургалтад** хэт урт дарааллыг (max 8K хүртэл) `MAX_LEN` урттай хэсгүүдэд хуваана. Хэсэг бүр шинэ $h_0$-оос эхэлнэ. Энэ нь GPU санах ойд багтаах түгээмэл арга бөгөөд өгүүлэлд энэ талаар бичээгүй. **Үнэлгээнд** дарааллыг таслахгүй, далд төлөвийг цонхноос цонх руу дамжуулна.
* Богино дарааллыг тэгээр дүүргэж (padding), `mask`-аар тэдгээрийг алдагдлаас хасна.

In [ ]:
MAX_LEN = 200

def chunk(seqs, max_len=MAX_LEN):
    out = []
    for q, a in seqs:
        for s in range(0, len(q), max_len):
            qq, aa = q[s:s + max_len], a[s:s + max_len]
            if len(qq) >= 2:
                out.append((qq, aa))
    return out

def make_batch(batch):
    T = max(len(q) for q, _ in batch) - 1
    B = len(batch)
    x = torch.zeros(B, T, dtype=torch.long)
    q_next = torch.zeros(B, T, dtype=torch.long)
    a_next = torch.zeros(B, T, dtype=torch.long)
    mask = torch.zeros(B, T, dtype=torch.bool)
    for i, (q, a) in enumerate(batch):
        n = len(q) - 1
        x[i, :n] = torch.from_numpy(q[:-1] + M * a[:-1])
        q_next[i, :n] = torch.from_numpy(q[1:])
        a_next[i, :n] = torch.from_numpy(a[1:])
        mask[i, :n] = True
    return x, q_next, a_next, mask

def batches(seqs, bs, shuffle):
    idx = np.random.permutation(len(seqs)) if shuffle else np.arange(len(seqs))
    for s in range(0, len(seqs), bs):
        yield make_batch([seqs[i] for i in idx[s:s + bs]])

### 3.2 Загвар

Өгүүллийн (1)–(2) тэгшитгэл: $\mathbf h_t = \text{RNN}(\mathbf x_t, \mathbf h_{t-1})$, $\;\mathbf y_t = \sigma(W_{yh}\,\text{dropout}(\mathbf h_t) + \mathbf b_y)$.
Dropout-ийг зөвхөн гаралт руу хэрэглэнэ, рекуррент холбоос руу хэрэглэхгүй (өгүүллийн 3.3-т заасны дагуу). PyTorch-ийн `nn.LSTM` дотроо dropout хийдэггүй тул энэ шаардлага шууд биелнэ.
Загвар $\sigma$-ийн өмнөх **logit**-ийг буцаана; $\sigma$-г алдагдлын функц дотор тооцно (`binary_cross_entropy_with_logits` тоон хувьд тогтвортой).

In [ ]:
class DKT(nn.Module):
    def __init__(self, M, hidden=200, cell="lstm", dropout=0.2):
        super().__init__()
        self.M = M
        rnn = {"lstm": nn.LSTM, "rnn": nn.RNN}[cell]
        self.rnn = rnn(2 * M, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, M)

    def forward(self, x_idx, state=None):
        x = F.one_hot(x_idx, 2 * self.M).float()      # (B, T, 2M) — өгүүллийн one-hot оролт
        h, state = self.rnn(x, state)                   # (B, T, H); state-ийг дараагийн хэсэгт дамжуулж болно
        return self.out(self.drop(h)), state            # (B, T, M) logit: бодлого бүрийг зөв бодох

m = DKT(M)
print(m, "\nпараметр:", sum(p.numel() for p in m.parameters()))

### 3.3 Алдагдлын функц — DKT-ийн гол санаа  ✍️ *(миний бичих хэсэг)*

Загвар алхам бүрт **$M$ бүх бодлогын** таамаглал гаргадаг, гэхдээ бид зөвхөн **дараагийн бодлого $q_{t+1}$**-ийн хариултыг мэднэ.
Өгүүллийн (3) тэгшитгэл:

$$\mathcal L = \sum_t \ell\big(\mathbf y_t^\top \delta(q_{t+1}),\; a_{t+1}\big)$$

$\mathbf y_t^\top \delta(q_{t+1})$ нь $\mathbf y_t$ векторын $q_{t+1}$ дугаартай элементийг сонгох үйлдэл. `logits` $(B,T,M)$-ээс `q_next` $(B,T)$-ийн заасан элементийг сонгоод, `mask`-аар padding-ийг хасаж, BCE тооцох хэрэгтэй.

In [ ]:
def dkt_loss(logits, q_next, a_next, mask):
    '''
    logits : (B, T, M) float — загварын гаралт, алхам бүрт бүх M бодлогын logit
    q_next : (B, T) long    — дараагийн бодлогын дугаар q_{t+1}
    a_next : (B, T) long    — дараагийн хариулт a_{t+1} (0/1)
    mask   : (B, T) bool    — True бол жинхэнэ алхам, False бол padding

    Буцаах: (loss, z)
      loss — mask-тай алхмуудын дундаж binary cross-entropy (скаляр)
      z    — mask-тай алхмуудын сонгосон logit, 1 хэмжээст (AUC тооцоход ашиглана)
    '''
    # TODO(Өсөхбаяр): өгүүллийн (3) тэгшитгэлийг 3-5 мөрөөр хэрэгжүүлэх.
    #   1) logits-оос q_next-ийн заасан элементийг сонгох  (зөвлөмж: torch.gather, dim=2)
    #   2) mask-аар padding алхмуудыг хасах
    #   3) F.binary_cross_entropy_with_logits(z, a.float())
    raise NotImplementedError("dkt_loss-ийг бичнэ үү")

### 3.4 Сургалт

Өгүүллийн тохиргоо: далд давхарга **200**, минибатч **100**, гаралтын dropout, gradient clipping.
Өгүүлэлд SGD гэж бичсэн боловч сурах хурд, momentum-ийг заагаагүй тул би **Adam (lr=1e-3)** хэрэглэв. Энэ нь өгүүллээс ялгаатай миний сонголт.
Validation AUC 3 эпох дараалан сайжрахгүй бол сургалтыг зогсоож, хамгийн сайн жинг сэргээнэ (early stopping).

In [ ]:
@torch.no_grad()
def dkt_predict(model, seqs, bs=64, win=MAX_LEN):
    # Үнэлгээнд дарааллыг таслахгүй: MAX_LEN цонхоор явахдаа далд төлөвийг дамжуулна.
    # Ингэснээр суурь загваруудтай яг ижил (q_{t+1}, a_{t+1}) зорилтууд дээр AUC тооцогдоно.
    model.eval()
    ys, ps = [], []
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i][0]))
    for s in range(0, len(order), bs):
        x, q, a, mask = make_batch([seqs[i] for i in order[s:s + bs]])
        state = None
        for w in range(0, x.shape[1], win):
            sl = slice(w, w + win)
            if not mask[:, sl].any():
                break
            logits, state = model(x[:, sl].to(DEVICE), state)
            _, z = dkt_loss(logits, q[:, sl].to(DEVICE), a[:, sl].to(DEVICE), mask[:, sl].to(DEVICE))
            ys.append(a[:, sl][mask[:, sl]].numpy()); ps.append(torch.sigmoid(z).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_dkt(train, val, cell="lstm", hidden=200, epochs=30, lr=1e-3, bs=100,
              dropout=0.2, clip=1.0, patience=3, seed=SEED, verbose=True):
    set_seed(seed)
    model = DKT(M, hidden, cell, dropout).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tr_chunks = chunk(train)
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot, n = 0.0, 0
        for x, q, a, mask in batches(tr_chunks, bs, shuffle=True):
            x, q, a, mask = x.to(DEVICE), q.to(DEVICE), a.to(DEVICE), mask.to(DEVICE)
            logits, _ = model(x)
            loss, _ = dkt_loss(logits, q, a, mask)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            tot += loss.item() * mask.sum().item(); n += mask.sum().item()
        auc = roc_auc_score(*dkt_predict(model, val))
        hist.append(dict(epoch=ep, train_loss=tot / n, val_auc=auc))
        if verbose:
            print(f"  эпох {ep:2d}  train BCE {tot / n:.4f}  val AUC {auc:.4f}  ({time.time() - t0:.1f} c)")
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

EPOCHS = 2 if QUICK else 30

In [ ]:
models, hists = {}, {}
for cell in ["lstm", "rnn"]:
    print(f"== DKT-{cell.upper()} ==")
    models[cell], hists[cell] = train_dkt(train_seqs, val_seqs, cell=cell, epochs=EPOCHS)
    results[f"DKT-{cell.upper()}"] = roc_auc_score(*dkt_predict(models[cell], test_seqs))
    print(f"DKT-{cell.upper()} test AUC = {results[f'DKT-{cell.upper()}']:.4f}\n")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for cell, h in hists.items():
    ax[0].plot(h.epoch, h.train_loss, "o-", label=cell.upper())
    ax[1].plot(h.epoch, h.val_auc, "o-", label=cell.upper())
ax[0].set_title("Train BCE"); ax[1].set_title("Validation AUC")
for a_ in ax: a_.set_xlabel("эпох"); a_.legend()
plt.tight_layout(); plt.show()

## 4. Үр дүн: өгүүлэлтэй харьцуулах

In [ ]:
paper = {"Marginal": 0.62, "BKT": 0.67, "DKT-LSTM": 0.86}
table = pd.DataFrame({"Миний test AUC": pd.Series(results).round(4),
                      "Өгүүлэл (Хүснэгт 1)": pd.Series(paper)})
table.loc["PFA", "Өгүүлэл (Хүснэгт 1)"] = np.nan
table.loc["DKT-RNN", "Өгүүлэл (Хүснэгт 1)"] = np.nan
table.sort_values("Миний test AUC")

**Анхаарах:** өгүүлэл 5-fold CV ашигласан, би зохиогчдын нэг train/test хуваалтыг ашиглав. Иймээс тоонууд яг ижил биш, харин **дараалал ба зөрүүний хэмжээ** харьцуулах гол зүйл.
Доорх нүд өгүүлэлтэй ижил 5-fold CV-г LSTM дээр давтана (T4 дээр ~10 мин).

In [ ]:
RUN_CV = not QUICK
if RUN_CV:
    everyone = train_seqs + val_seqs + test_seqs
    folds = np.array_split(np.random.default_rng(SEED).permutation(len(everyone)), 5)
    cv = []
    for i, te_idx in enumerate(folds):
        te = [everyone[j] for j in te_idx]
        rest = [everyone[j] for j in np.concatenate([f for k, f in enumerate(folds) if k != i])]
        va, tr = rest[: len(rest) // 10], rest[len(rest) // 10:]
        mdl, _ = train_dkt(tr, va, cell="lstm", epochs=EPOCHS, verbose=False)
        row = {"fold": i + 1, "DKT-LSTM": roc_auc_score(*dkt_predict(mdl, te))}
        row["Marginal"] = roc_auc_score(*marginal_predict(tr + va, te))
        row["PFA"] = roc_auc_score(*pfa_predict(tr + va, te))
        cv.append(row); print(row)
    cv = pd.DataFrame(cv).set_index("fold")
    display(cv.agg(["mean", "std"]).round(4))

## 5. Нэмэлт судалгаа

### 5a. Өгөгдлийн хэмжээ: DKT хэдэн хариултаас суурь загваруудыг давах вэ?

Өгүүлэлд *«RNN-д их өгөгдөл хэрэгтэй, жижиг ангид тохиромжгүй»* гэж бичсэн ч хэр их гэдгийг тоогоор хэлээгүй.
Сургалтын сурагчдын $5\%, 10\%, 25\%, 50\%, 100\%$-ийг авч, загвар бүрийг сургаад **ижил test** дээр үнэлнэ.
Энэ нь `pimath.mn`-д DKT руу шилжих өгөгдлийн босгыг тооцоход хэрэгтэй.

In [ ]:
FRACS = [0.05, 0.25] if QUICK else [0.05, 0.1, 0.25, 0.5, 1.0]
curve = []
for frac in FRACS:
    k = max(20, int(len(train_seqs) * frac))
    sub = [train_seqs[i] for i in np.random.default_rng(SEED).permutation(len(train_seqs))[:k]]
    row = {"frac": frac, "сурагч": k, "хариулт": n_ans(sub)}
    row["Marginal"] = roc_auc_score(*marginal_predict(sub, test_seqs))
    row["PFA"] = roc_auc_score(*pfa_predict(sub, test_seqs))
    row["BKT"] = roc_auc_score(*bkt_predict(sub, test_seqs))
    mdl, _ = train_dkt(sub, val_seqs, cell="lstm", epochs=EPOCHS, verbose=False)
    row["DKT-LSTM"] = roc_auc_score(*dkt_predict(mdl, test_seqs))
    curve.append(row); print({k_: (round(v, 4) if isinstance(v, float) else v) for k_, v in row.items()})
curve = pd.DataFrame(curve)

plt.figure(figsize=(7, 4))
for name in ["Marginal", "BKT", "PFA", "DKT-LSTM"]:
    plt.plot(curve["хариулт"], curve[name], "o-", label=name)
plt.xscale("log"); plt.xlabel("сургалтын хариултын тоо (log)"); plt.ylabel("test AUC")
plt.title("Өгөгдлийн хэмжээ ба AUC"); plt.legend(); plt.grid(alpha=.3); plt.show()
curve.round(4)

### 5b. Бодлого хоорондын нөлөө $J_{ij}$ (өгүүллийн (4) тэгшитгэл)

$J_{ij} = y(j\mid i) / \sum_k y(j\mid k)$: эхний алхамд $i$-г зөв бодсон сурагч хоёр дахь алхамд $j$-г зөв бодох таамагласан магадлалыг бүх $i$-ээр нормчилно.
Хамгийн их хариулттай 25 чадвар дээр дулааны зураг ба хамгийн хүчтэй 10 холбоог харуулна.

In [ ]:
@torch.no_grad()
def influence_matrix(model):
    model.eval()
    x = (torch.arange(M) + M * 1).unsqueeze(1).to(DEVICE)   # (M, 1): i-г зөв бодсон нэг алхам
    Y = torch.sigmoid(model(x)[0])[:, 0, :].cpu().numpy()  # Y[i, j] = y(j | i)
    return Y / Y.sum(axis=0, keepdims=True)                # багана бүрийг Σ_k-ээр нормчилно

J = influence_matrix(models["lstm"])
top = np.argsort(-skill_cnt)[:25]
plt.figure(figsize=(7, 6))
plt.imshow(J[np.ix_(top, top)], cmap="viridis")
plt.xticks(range(25), top, rotation=90, fontsize=7); plt.yticks(range(25), top, fontsize=7)
plt.xlabel("j (дараагийн бодлого)"); plt.ylabel("i (зөв бодсон бодлого)")
plt.title("Нөлөөний матриц J_ij (хамгийн их хариулттай 25 чадвар)"); plt.colorbar(); plt.show()

Joff = J.copy(); np.fill_diagonal(Joff, 0); Joff[skill_cnt == 0] = 0; Joff[:, skill_cnt == 0] = 0
flat = np.argsort(-Joff, axis=None)[:10]
# J_ij-ийн жигд утга 1/M ≈ 0.008 тул J·M нь «жигдээс хэд дахин их» гэдгийг харуулна
pd.DataFrame([(i, j, round(Joff[i, j], 4), round(Joff[i, j] * M, 2))
              for i, j in zip(*np.unravel_index(flat, J.shape))],
             columns=["i", "j", "J_ij", "J·M"])

### 5c. Нэг сурагчийн мэдлэгийн явц (өгүүллийн Зураг 1-тэй адил)

Test-ийн нэг сурагчийн эхний 50 хариулт дээр загварын таамагласан магадлалыг тухайн сурагчийн бодсон чадваруудаар харуулна. ● зөв, ○ буруу.

In [ ]:
@torch.no_grad()
def trajectory(model, q, a, steps=50):
    model.eval()
    q, a = q[:steps], a[:steps]
    x = torch.from_numpy(q + M * a).unsqueeze(0).to(DEVICE)
    return torch.sigmoid(model(x)[0])[0].cpu().numpy(), q, a   # (T, M)

cand = [s for s in test_seqs if len(s[0]) >= 50 and 3 <= len(np.unique(s[0][:50])) <= 6]
q, a = cand[0]
P, q, a = trajectory(models["lstm"], q, a)
skills = list(dict.fromkeys(q.tolist()))
plt.figure(figsize=(12, 0.5 * len(skills) + 1.2))
plt.imshow(P[:, skills].T, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
for t in range(len(q)):
    plt.scatter(t, skills.index(q[t]), marker="o", s=30,
                facecolors="k" if a[t] else "none", edgecolors="k")
plt.yticks(range(len(skills)), [f"чадвар {s}" for s in skills])
plt.xlabel("алхам t"); plt.title("Таамагласан P(зөв) — алхам бүрийн дараа"); plt.colorbar(); plt.show()

## 6. Дүгнэлт

*(Сургалт дууссаны дараа бодит тоонуудаар бөглөнө.)*

1. **Давтагдах байдал:** …
2. **Суурь загвартай харьцуулалт:** …
3. **Өгөгдлийн хэмжээ:** …
4. **pimath.mn-д хэрэглэх нь:** …